# ಪಾಠ 11 - ಮodel Context Protocol (MCP)

**ಮodel Context Protocol (MCP)** ಒಂದು ತೆರೆಯಲ್ಪಟ್ಟ ಮಾನದಂಡವಾಗಿದ್ದು, ಅದು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಪರಿಕರಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಡೇಟಾ ಮೂಲಗಳನ್ನು ಚಾಲನೆ ಸಮಯದಲ್ಲಿ ಡೈನಾಮಿಕ್‌ವಾಗಿ ಕಂಡು ಹಿಡಿದು ಉಪಯೋಗಿಸಲು ಅವಕಾಶ ಮಾಡಿಕೊಡುತ್ತದೆ. ಏಜೆಂಟ್‌ಗೆ ಕಠಿಣ ಕೋಡ್ ಮಾಡಬೇಕಾದ ಪರಿಕರಗಳ ಬದಲು, MCP ಏಜೆಂಟ್‌ಗಳಿಗೆ ಬೇರೆ ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕ ಹೊಂದಲು ಸಾಧ್ಯವಾಗುತ್ತದೆ, ಅವುಗಳಲ್ಲಿ ಅವಶ್ಯಕತೆ ಇದ್ದಾಗ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಬಹಿರಂಗಪಡಿಸಲಾಗುತ್ತದೆ.

ಈ ಪಾಠದಲ್ಲಿ, ನೀವು ಕಲಿಯುವಿರಿ:
- MCP ಏನು ಮತ್ತು ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳಿಗಾಗಿಯೇ ಅದಕ್ಕೆ ಮಹತ್ವವೇನು
- MCP ಕ್ಲೈಂಟ್-ಸರ್ವರ್ ವಾಸ್ತುಶಿಲ್ಪ ಹೇಗೆ ಕೆಲಸ ಮಾಡುತ್ತದೆ
- MCP ಶೈಲಿಯ ಪರಿಕರ ಕಂಡುಹಿಡಿತನ್ನು ಬಳಸುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸುವುದು


## ಸೆಟಅಪ್

**ಮುಂದಡೆ ಅವಶ್ಯಕತೆಗಳು:**
- ನಿಯೋಜಿಸಲಾದ ಮಾದರಿಯೊಂದಿಗೆ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್
- ದೃಢೀಕರಣಕ್ಕಾಗಿ `az login` ಅನ್ನು ರನ್ ಮಾಡಿ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಮಾದರಿ ಸನ್ನಿವೇಶ ಪ್ರೋಟೋಕಾಲ್ (MCP) ಎಂದರೆ ಏನು?

MCP ಸಹಾಯದಿಂದ AI ಏಜೆಂಟ್‌ಗಳು ಬಾಹ್ಯ ಸಾಧನಗಳು ಮತ್ತು ಡೇಟಾ ಮೂಲಗಳನ್ನು ಕಂಡುಹಿಡಿದು ಸಬಂಧಿಸಲು ಒಂದು ಪ್ರಮಾಣಿತ ವಿಧಾನವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತದೆ:

- **MCP ಸರ್ವರ್**: ಸಾಧನಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ಒಂದು ಪ್ರಮಾಣಿತ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಪ್ರದರ್ಶಿಸುತ್ತದೆ
- **MCP ಕ್ಲೈಂಟ್**: ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕಿಸುವ ಮತ್ತು ಲಭ್ಯವಿರುವ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಕಂಡುಹಿಡಿಯುವ ಏಜೆಂಟ್ ರನ್‌ಟೈಮ್
- **ಡೈನಾಮಿಕ್ ಕಂಡುಹಿಡಿತ**: ಏಜೆಂಟ್‌ಗಳಿಗೆ ಕಠಿಣವಾಗಿ ಸಂರಚಿತ ಸಾಧನಗಳ ಅಗತ್ಯವಿಲ್ಲ — ಅವು ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಲಭ್ಯವಿರುವವುವನ್ನು ಕಂಡುಹಿಡಿಯೋದು

ಇದು ಏಜೆಂಟ್ ಕೋಡ್ ಬದಲಾವಣೆ ಮಾಡದೆ ಹೊಸ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಸೇರಿಸಬಹುದಾದ ವಿಸ್ತರಣೀಯ ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಲು ಶಕ್ತಿದಾಯಕವಾಗಿದೆ.


## MCP ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ಏಜೆಂಟ್ (MCP ಕ್ಲೈಂಟ್) MCP ಸರ್ವರ್‌ಗಾಗಿ ಸಂಪರ್ಕ ಹೊಂದುತ್ತದೆ
2. ಸರ್ವರ್ ಲಭ್ಯವಿರುವ ಸಾಧನಗಳ ತಿದ್ದುಪಡಿಗಳು ಮತ್ತು ಅವುಗಳ ವೈವಿಧ್ಯಗಳ ಪಟ್ಟಿಯನ್ನು ಪ್ರತಿಕ್ರಿಯಿಸುತ್ತದೆ
3. ನಂತರ ಏಜೆಂಟ್ ತನ್ನ ತರ್ಕದ ಸಮಯದಲ್ಲಿ ಯಾವುದೇ ಕಂಡುಹಿಡಿದ ಸಾಧನವನ್ನು ಕರೆಮಾಡಬಹುದು
4. ಫಲಿತಾಂಶಗಳು ಅದೇ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಹಿಂತಿರುಗುತ್ತವೆ


## MCP ಸಾಧನ ಪತ್ತೆಹಚ್ಚುವಿಕೆಯ ಅನುಕರಣೆ

ನಿಜವಾದ MCP ಸರ್ವರ್‌ಗೆ ನಡೆಸಲಾಗುತ್ತಿರುವ ಸರ್ವರ್ ಪ್ರಕ್ರಿಯೆಯ आवश्यकता ಇರುವುದರಿಂದ, ನಾವು MCP-ಸಂಬಂಧಿತ ವಸತಿ ಸೇವೆ ನೀಡುವ `@tool` ಕಾರ್ಯಗಳನ್ನು ಅನುಕರಿಸಿ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತೇವೆ.

ಉತ್ಪಾದನೆಗೆ, ಈ ಸಾಧನಗಳು ಸ್ಥಳೀಯವಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸದೇ, MCP ಸರ್ವರ್‌ನಿಂದ ಗತಿಸ್ಥಿತಿಯಲ್ಲಿ ಪತ್ತೆಹಚ್ಚಲಾಗುತ್ತವೆ.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ಶೈಲಿಯ ಟೂಲ್ಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ನಿರ್ಮಾಣ


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ಉದ್ಭವನದಲ್ಲಿ MCP

ಉತ್ಪಾದನಾ ವಾತಾವರಣದಲ್ಲಿ, MCP ಶಕ್ತಿಶಾಲಿ ಮಾದರಿಗಳನ್ನು ಸಾಧ್ಯ ಮಾಡುತ್ತದೆ:

- **ಡೈನಾಮಿಕ್ ಟೂಲ್ ಕಂಡುಕೊಳ್ಳುವಿಕೆ**: ಏಜೆಂಟ್ಗಳು MCP ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕ ಮಾಡಿ ಟೂಲ್ಗಳನ್ನು ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಕಂಡುಹಿಡಿಯುತ್ತಾರೆ
- **ವಿಚ್ಛೇದಿತ ವಾಸ್ತುಶಿಲ್ಪ**: ಟೂಲ್ ಒದಗಿಸುವವರು ಏಜೆಂಟ್‌ನಿಂದ ಸ್ವತಂತ್ರವಾಗಿ ನವೀಕರಿಸಬಹುದು
- **ಕ್ರಾಸ್-ಸಂಸ್ಥೆ ಹಂಚಿಕೆ**: ತಂಡಗಳು ಯಾವುದೇ ಏಜೆಂಟ್ ಉಪಯೋಗಿಸಬಹುದಾದ ಸಾಮರ್ಥ್ಯಗಳನ್ನು MCP ಸರ್ವರ್‌ಗಳ ಮೂಲಕ ಅನಾವರಣ ಮಾಡಬಹುದು
- **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಬೆಂಬಲ**: MAF ನ್ನು `mcp` ಇಂಟಿಗ್ರೇಶನ್ ಮೂಲಕ ಬಿಲ್ಟ್-ಇನ್ MCP ಕ್ಲೈಂಟ್ ಬೆಂಬಲವಿದೆ

ನಿಜವಾದ MCP ಸರ್ವರ್ ಅನ್ನು MAF ನೊಂದಿಗೆ ಬಳಸಲು, ನೀವು `hosted_mcp_tool()` ಅಥವಾ MCP ಕ್ಲೈಂಟ್ ಇಂಟಿಗ್ರೇಶನ್ ಮೂಲಕ ಸಂಪರ್ಕಿಸಬಹುದು.

**ಹೆಚ್ಚಾಗಿ ತಿಳಿದುಕೊಳ್ಳಿ:**
- [MCP ವಿವರಣೆ](https://modelcontextprotocol.io/)
- [ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ MCP ಬೆಂಬಲ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ, ನೀವು ಕಲಿದರು:
- **MCP** ಎಂದರೆ ಏಜೆಂಟ್‌ಗಳು ಮತ್ತು ಟೂಲ್ ಒದಗಿಸುವವರ ನಡುವೆ ಡೈನಾಮಿಕ್ ಟೂಲ್ ಕಂಡುಹಿಡಿಯುವುದಕ್ಕಾಗಿ ఓಪನ್ ಸ್ಟ್ಯಾಂಡರ್ಡ್
- **ಕ್ಲೈಂಟ್-ಸರ್ವರ್ ವಾಸ್ತುಶಿಲ್ಪ** ಏಜೆಂಟ್‌ಗಳಿಗೆ ಕಾರ್ಯಾವಧಿಯಲ್ಲಿ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಕಂಡುಹಿಡಿಯಲು ಅನುಮತಿಸುತ್ತದೆ
- MCP **ವಿಸ್ತಾರಗೊಳ್ಳಬಹುದಾದ, Decoupled ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು** ಸಕ್ರಿಯಗೊಳಿಸುತ್ತದೆ ಅಲ್ಲಿ ಕೋಡ್ ಬದಲಾವಣೆಗಳಿಲ್ಲದೆ ಸಾಧನಗಳನ್ನು ಸೇರಿಸಬಹುದು
- ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಉತ್ಪಾದನಾ ಬಳಕೆಗೆ **ನಿರ್ಮಿತ MCP ಬೆಂಬಲವನ್ನು** ಒದಗಿಸುತ್ತದೆ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
